In [ ]:

import warnings

import numpy as np
import pandas as pd

import torch
import torchvision
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
from torchvision.transforms import v2
import matplotlib.pyplot as plt

np.set_printoptions(suppress=True, precision=5, linewidth=1000)
pd.options.mode.chained_assignment = None
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option("display.precision", 5)
warnings.filterwarnings("ignore")

train_data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([ToTensor(), v2.ToDtype(torch.float, scale=True)])
)

valid_data = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform= v2.Compose([ToTensor(), v2.ToDtype(torch.float, scale=True)])
)

valid_data, test_data = torch.utils.data.random_split(valid_data, [9000, 1000])

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=True)



1024


Display a batch of images:

In [ ]:

import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import ImageGrid

X, y = next(iter(train_loader))

# fig, ax = plt.subplots(1, 1, figsize=(8, 8), tight_layout=True)
# ax.axis("off")
# ax.set_title("Training Images", fontsize=10)
# ax.imshow(np.transpose(torchvision.utils.make_grid(X, padding=6, normalize=False), (1, 2, 0)))
# plt.show()

fig = plt.figure(figsize=(10., 10.), tight_layout=False)

grid = ImageGrid(fig, 111, nrows_ncols=(8, 8), axes_pad=0.25)

for ax, X_ii, y_ii in zip(grid, X, y):
    X_ii = np.transpose(X_ii.numpy(), (1, 2, 0))
    ax.imshow(X_ii, cmap="gray")
    ax.axis("off")
    ax.set_title(f"{y_ii}", fontsize=8)

plt.show()



<br>

Define model architecture:

In [ ]:

class MLP(nn.Module):
    
    def __init__(self, dropout=0.0):
        
        super().__init__()

        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=28 * 28, out_features=128),
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(in_features=128, out_features=10)
        )

    def forward(self, X):
        output = self.model(X)
        return output


nbr_params = sum(p.numel() for p in MLP().parameters())
print(f"MLP number of parameters: {nbr_params:,.0f}")


In [120]:

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

model = MLP(dropout=0.10).to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=1e-3, momentum=.90)

loss_fn = nn.CrossEntropyLoss()

print(device)


cuda:0


Training loop:

- Load a batch of training data from the DataLoader.
- Zero out the optimizer’s gradients.
- Get predictions from the model for current batch.
- Calculate the loss for current batch predictions vs. labels.
- Calculate backward gradients over the weights.
- Calculate the loss on a set of data that was not used for training.

In [124]:


def epoch_trainer(epoch, data_loader, model, loss_fn, optimizer):
    """
    Execute a single training epoch. Return last batch training loss
    and accuracy. 
    """
    loss, checkpoint_loss, correct, samples = 0.0, 0.0, 0, 0

    # Put model in train mode.
    model.train()

    # Iterate over batches in data_loader. 
    for ii, (X, yactual) in enumerate(data_loader):

        # Send datasets to device. 
        X, yactual = X.to(device), yactual.to(device)

        # Zero out gradients.
        optimizer.zero_grad()

        # Forward pass. 
        ypred = model(X)

        # Compute loss. 
        loss_ii = loss_fn(ypred, yactual)

        # Backpropagation and optimizer step. 
        loss_ii.backward()
        optimizer.step()

        # Update running_loss.
        loss+=loss_ii.item()
        correct+=(ypred.argmax(dim=1)==yactual).type(torch.float).sum().item()
        samples+=yactual.size(dim=0)

        # Print running_loss for every 1000 mini-batches.
        if ii % 250 == 0:
            checkpoint_acc = correct / samples
            checkpoint_loss = loss / 250

            print(f"\t+ [train][epoch={epoch}, batch={ii}] loss = {checkpoint_loss:,.5f}, acc = {checkpoint_acc:.5f}.")
            
            loss, correct, samples = 0.0, 0, 0

    return checkpoint_loss, checkpoint_acc
        


def epoch_validator(data_loader, model, loss_fn):
    """
    Execute a single validation epoch. Return average validation loss
    and accuracy.
    """
    valid_loss, correct = 0.0, 0

    # Put model in validation mode.
    model.eval()

    with torch.no_grad():

        for ii, (X, yactual) in enumerate(data_loader, start=1):

            # Send dataset and target to device. 
            X, yactual = X.to(device), yactual.to(device)

            # Get model predictions. 
            ypred = model(X)

            # Compute loss and update valid_loss.
            valid_loss+=loss_fn(ypred, yactual).item()

            # Count number of correct class predictions.
            correct+=(ypred.argmax(dim=1)==yactual).type(torch.float).sum().item()

    loss, acc = valid_loss / len(data_loader), correct / len(data_loader.dataset)

    return loss, acc


In [104]:

model = MLP().to("cpu")

yact = y.cpu()
ypred = model(X.cpu())



In [125]:

n_epochs = 10

results = []

for epoch in range(1, n_epochs + 1):

    tloss, tacc = epoch_trainer(
        epoch=epoch, data_loader=train_loader, model=model, loss_fn=loss_fn, 
        optimizer=optimizer
    )
    
    vloss, vacc = epoch_validator(
        data_loader=valid_loader, model=model, loss_fn=loss_fn
    )
    
    print(f"[epoch={epoch}]: tloss={tloss:.5f}, tacc={tacc:.5f}, vloss={vloss:.5f}, vacc={vacc:.5f}.")

    # Append metrics to results.
    results.append((tloss, tacc, vloss, vacc))


	+ [train][epoch=1, batch=0] loss = 0.00291, acc = 0.70312.
	+ [train][epoch=1, batch=250] loss = 0.48564, acc = 0.86887.
	+ [train][epoch=1, batch=500] loss = 0.44902, acc = 0.87681.
	+ [train][epoch=1, batch=750] loss = 0.42793, acc = 0.87781.
[epoch=1]: tloss=0.42793, tacc=0.87781, vloss=0.36675, vacc=0.90044.
	+ [train][epoch=2, batch=0] loss = 0.00148, acc = 0.92188.
	+ [train][epoch=2, batch=250] loss = 0.40366, acc = 0.88569.
	+ [train][epoch=2, batch=500] loss = 0.39741, acc = 0.88831.
	+ [train][epoch=2, batch=750] loss = 0.37995, acc = 0.89487.
[epoch=2]: tloss=0.37995, tacc=0.89487, vloss=0.33222, vacc=0.90722.
	+ [train][epoch=3, batch=0] loss = 0.00133, acc = 0.87500.
	+ [train][epoch=3, batch=250] loss = 0.37099, acc = 0.89444.
	+ [train][epoch=3, batch=500] loss = 0.35335, acc = 0.89925.
	+ [train][epoch=3, batch=750] loss = 0.35875, acc = 0.89819.
[epoch=3]: tloss=0.35875, tacc=0.89819, vloss=0.30742, vacc=0.91289.
	+ [train][epoch=4, batch=0] loss = 0.00082, acc = 0.95

In [126]:
results


[(0.4279320111274719, 0.8778125, 0.3667549129496229, 0.9004444444444445),
 (0.3799536615610123, 0.894875, 0.33222326171313615, 0.9072222222222223),
 (0.3587501368522644, 0.8981875, 0.3074211283778468, 0.9128888888888889),
 (0.3264698356986046, 0.9075, 0.2909381318282574, 0.918),
 (0.31717215833067897, 0.9100625, 0.2781853015329821, 0.921),
 (0.29983772015571597, 0.9146875, 0.26387537519137066, 0.9255555555555556),
 (0.2853497519791126, 0.916875, 0.25336152214741875, 0.9292222222222222),
 (0.2797610825300217, 0.9196875, 0.24213411600877208, 0.9315555555555556),
 (0.26458981496095657, 0.92375, 0.23410979975411234, 0.9326666666666666),
 (0.257844920784235, 0.92675, 0.22417577482918474, 0.9368888888888889)]

In [97]:

loss, acc = epoch_validator(
    data_loader=test_loader, model=model, loss_fn=loss_fn
)

print(f"Final loss     : {loss:.3f}")
print(f"Final accuracy : {acc:.3f}")


Final loss     : 0.257
Final accuracy : 0.926


In [ ]:

# Assess performance on holdout data.

preds = mdl2(X).argmax(dim=1)

nbr_correct = (y == preds).sum()

print(f"Number correctly classified instances: {nbr_correct}/{X.shape[0]}")


In [84]:

from torcheval.metrics.functional import multiclass_confusion_matrix


ModuleNotFoundError: No module named 'torcheval'